# 🤖 Model Training

## Enterprise AutoML Platform

## 📖 About this Notebook

This notebook demonstrates training multiple machine learning models and comparing their performance using standard evaluation metrics.

The best-performing model is selected automatically based on evaluation results.

Sections

1. Title
2. Objective
3. Import Libraries
4. Load Dataset
5. Data Preprocessing
6. Train Test Split
7. Build Models
8. Train Models
9. Evaluate Models
10. Create Leaderboard
11. Best Model Selection
12. Save Best Model
13. Conclusion

### Objective

The objective of this notebook is to train multiple machine learning models,
compare their performance and automatically identify the best-performing model.

Models Included

- Logistic Regression
- Random Forest
- XGBoost
- LightGBM
- CatBoost


Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

Load Dataset

In [ ]:
df = pd.read_csv(
    "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df.head()

Encode Target

In [ ]:
df["Churn"] = df["Churn"].map(
    {
        "No":0,
        "Yes":1,
    }
)

Split Features

In [ ]:
X = df.drop(
    columns=["Churn"]
)

y = df["Churn"]

Numerical Columns

In [ ]:
numerical_columns = X.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

Categorical Columns

In [ ]:
categorical_columns = X.select_dtypes(
    include=["object","category","bool"]
).columns.tolist()

Train Test Split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

Preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(strategy="median"),
                    ),
                    (
                        "scaler",
                        StandardScaler(),
                    ),
                ]
            ),
            numerical_columns,
        ),
        (
            "cat",
            Pipeline(
                [
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="most_frequent",
                        ),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=False,
                        ),
                    ),
                ]
            ),
            categorical_columns,
        ),
    ]
)

Transform Dataset

In [ ]:
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

Models

In [ ]:
models = {

    "Logistic Regression":
    LogisticRegression(
        max_iter=1000,
    ),

    "Random Forest":
    RandomForestClassifier(
        random_state=42,
    ),

    "XGBoost":
    XGBClassifier(
        eval_metric="logloss",
        verbosity=0,
        random_state=42,
    ),

    "LightGBM":
    LGBMClassifier(
        random_state=42,
        verbose=-1,
    ),

    "CatBoost":
    CatBoostClassifier(
        random_state=42,
        verbose=False,
    ),
}

Train Models

In [ ]:
leaderboard = []

best_model = None

best_score = 0

In [ ]:
for name,model in models.items():

    model.fit(
        X_train,
        y_train,
    )

    prediction = model.predict(
        X_test,
    )

    accuracy = accuracy_score(
        y_test,
        prediction,
    )

    precision = precision_score(
        y_test,
        prediction,
    )

    recall = recall_score(
        y_test,
        prediction,
    )

    f1 = f1_score(
        y_test,
        prediction,
    )

    leaderboard.append(
        {
            "Model":name,
            "Accuracy":accuracy,
            "Precision":precision,
            "Recall":recall,
            "F1":f1,
        }
    )

    if accuracy > best_score:

        best_score = accuracy

        best_model = model

Leaderboard

In [ ]:
leaderboard = pd.DataFrame(
    leaderboard
)

leaderboard = leaderboard.sort_values(
    by="Accuracy",
    ascending=False,
)

leaderboard

Best Model

In [ ]:
leaderboard.iloc[0]

Save Model

In [ ]:
joblib.dump(
    best_model,
    "../saved_models/best_model.pkl",
)

Save Leaderboard

In [ ]:
leaderboard.to_csv(
    "../reports/model_leaderboard.csv",
    index=False,
)

Conclusion

# ✅ Conclusion

Successfully completed:

- Multi-model training
- Model comparison
- Automatic leaderboard generation
- Best model selection
- Model serialization

The trained model is now ready for hyperparameter optimization.

---

# 🏭 Production Implementation

The production implementation uses the Enterprise AutoML Engine.

Core production modules include:

- Model Factory
- Model Trainer
- Metrics Engine
- Model Evaluation
- Leaderboard Generation

These modules automate the complete model training workflow.